# Traffic Violation Detection — Colab T4

这个 Notebook 用于在 Google Colab 免费 T4 上复现项目，并验证真实道路视频。按顺序运行全部单元格即可。

第一轮验证暂时关闭 PaddleOCR：Colab 当前 Python 3.13 与 PaddlePaddle 的兼容性不稳定，车牌识别结果会明确记录为 `UNKNOWN`，不会生成模拟号码。

In [ ]:
# 0. 检查运行时：应显示 NVIDIA T4。若没有，请选择 运行时/代码执行程序 → 更改运行时类型 → T4 GPU。
import os, platform, subprocess, sys
print('Python:', sys.version)
print('Platform:', platform.platform())
subprocess.run(['nvidia-smi'], check=False)

In [ ]:
# 1. 克隆或更新项目
from pathlib import Path
import os, subprocess

REPO_URL = 'https://github.com/mengdie305-hue/traffic-violation-detection.git'
PROJECT_DIR = Path('/content/traffic-violation-detection')
if (PROJECT_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
print('Project:', PROJECT_DIR)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)

In [ ]:
# 2. 安装 Colab 推理依赖（不重装 Colab 自带的 torch）
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)

import cv2, torch, ultralytics
print('torch:', torch.__version__)
print('opencv:', cv2.__version__)
print('ultralytics:', ultralytics.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('没有检测到 GPU，请先把运行时改成 T4 GPU。')

In [ ]:
# 3. 下载 YOLOv8n 权重到项目配置所需位置
from pathlib import Path
import shutil
from ultralytics import YOLO

model_dir = Path('data/models')
model_dir.mkdir(parents=True, exist_ok=True)
target_model = model_dir / 'yolov8n.pt'
if not target_model.exists():
    YOLO('yolov8n.pt')
    shutil.copy2('yolov8n.pt', target_model)
print('Model:', target_model.resolve(), target_model.stat().st_size, 'bytes')

In [ ]:
# 4. 运行自动化测试
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)

In [ ]:
# 5. 先跑 Mock 闭环，确认检测→追踪→判定→日志→视频全部正常
import subprocess, sys
subprocess.run([sys.executable, 'scripts/generate_test_video.py'], check=True)
subprocess.run([
    sys.executable, 'src/main.py', '--mock',
    '--video', 'data/videos/test_intersection.mp4',
    '--output', 'data/videos/colab_mock_output.mp4',
], check=True)

In [ ]:
# 6. 预览 Mock 输出并查看最新日志
from IPython.display import Video, display
from pathlib import Path
import pandas as pd, subprocess

mock_source = Path('data/videos/colab_mock_output.mp4')
mock_preview = Path('data/videos/colab_mock_output_h264.mp4')
subprocess.run([
    'ffmpeg', '-y', '-loglevel', 'error', '-i', str(mock_source),
    '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-movflags', '+faststart',
    str(mock_preview),
], check=True)
display(Video(str(mock_preview), embed=True, width=800))
logs = sorted(Path('data/logs').glob('violations_*.csv'), key=lambda p: p.stat().st_mtime)
if logs:
    print('Latest log:', logs[-1])
    display(pd.read_csv(logs[-1]))

## 获取真实视频

默认自动下载 Zenodo 上的 `video6.mp4`（CC BY 4.0），它是约 8 秒的固定机位路口片段，红灯、停止线和车辆均可见，适合先验证真实 YOLO + DeepSORT 流程。论文使用时请引用：Tiago Silva, *Dataset for Traffic Evaluation*, v2, Zenodo, DOI: 10.5281/zenodo.14041920。

也可以把 `USE_ZENODO_SAMPLE` 改成 `False`，上传其他有授权的固定机位路口视频。第一轮建议 30 秒到 3 分钟、720p 或 1080p，画面中最好同时看得到停止线和交通灯。

In [ ]:
# 7. 下载公开闯红灯样本；需要自有视频时把开关改为 False
USE_ZENODO_SAMPLE = True
ZENODO_URL = 'https://zenodo.org/api/records/14041920/files/video6.mp4/content'
EXPECTED_MD5 = 'caa146f7ba10f2081c4922f42d9a42f8'

from google.colab import files
from pathlib import Path
import hashlib, shutil, urllib.request

real_video = Path('data/videos/real_intersection.mp4')
real_video.parent.mkdir(parents=True, exist_ok=True)
if USE_ZENODO_SAMPLE:
    if not real_video.exists() or hashlib.md5(real_video.read_bytes()).hexdigest() != EXPECTED_MD5:
        print('Downloading CC BY 4.0 sample from Zenodo...')
        urllib.request.urlretrieve(ZENODO_URL, real_video)
    actual_md5 = hashlib.md5(real_video.read_bytes()).hexdigest()
    if actual_md5 != EXPECTED_MD5:
        raise RuntimeError(f'MD5 mismatch: {actual_md5}')
    print('Dataset citation: Silva, Tiago (2024), Dataset for Traffic Evaluation v2,')
    print('https://doi.org/10.5281/zenodo.14041920 (CC BY 4.0)')
else:
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('未选择视频。')
    source_name = next(iter(uploaded))
    shutil.move(source_name, real_video)
print('Real video:', real_video.resolve(), real_video.stat().st_size, 'bytes')

In [ ]:
# 8. 查看第一帧尺寸，下一格需要按图修改停止线和交通灯 ROI
import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(str(real_video))
ok, frame = cap.read()
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
if not ok:
    raise RuntimeError('视频无法解码。')
h, w = frame.shape[:2]
print(f'Original: {w}x{h}, FPS={fps:.2f}, frames={frame_count}, duration={frame_count/fps:.1f}s')
print('Pipeline 会缩放到 1280x720，请按下图的 1280x720 坐标设置 ROI。')
preview = cv2.resize(frame, (1280, 720))
plt.figure(figsize=(16, 9))
plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
plt.xlim(0, 1280); plt.ylim(720, 0); plt.grid(); plt.show()

In [ ]:
# 9. 依据上图修改这两个坐标，然后生成 Colab 配置
STOP_LINE = [520, 530, 1240, 455]       # video6: 近端车流停止线 [x1, y1, x2, y2]
TRAFFIC_LIGHT_BBOX = [810, 45, 860, 120] # video6: 中间上方红灯 [x1, y1, x2, y2]

from pathlib import Path
import yaml
with open('configs/config.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
cfg['roi']['stop_line'] = STOP_LINE
cfg['roi']['traffic_light_bbox'] = TRAFFIC_LIGHT_BBOX
cfg['violation']['red_light']['crossing_direction'] = 'upward'
cfg['violation']['red_light']['enabled'] = True
cfg['violation']['speeding']['enabled'] = False
cfg['violation']['wrong_way']['enabled'] = False
cfg['traffic_light']['source'] = 'configured_roi'
cfg['traffic_light']['brightness_threshold'] = 30
cfg['traffic_light']['min_area_ratio'] = 0.003
cfg['recognition']['backend'] = 'disabled'
cfg['video']['frame_skip'] = 1
cfg['video']['output_path'] = 'data/videos/real_intersection_output.mp4'
with open('configs/config_colab.yaml', 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)
print(Path('configs/config_colab.yaml').read_text())

In [ ]:
# 10. 使用 T4 运行真实 YOLO + DeepSORT 推理
import subprocess, sys
subprocess.run([
    sys.executable, 'src/main.py',
    '--video', str(real_video),
    '--config', 'configs/config_colab.yaml',
    '--output', 'data/videos/real_intersection_output.mp4',
    '--device', 'cuda',
], check=True)

In [ ]:
# 11. 预览结果、查看最新日志，并打包下载
from IPython.display import Video, display
from pathlib import Path
from google.colab import files
import pandas as pd, shutil, subprocess

output_video = Path('data/videos/real_intersection_output.mp4')
preview_video = Path('data/videos/real_intersection_output_h264.mp4')
subprocess.run([
    'ffmpeg', '-y', '-loglevel', 'error', '-i', str(output_video),
    '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-movflags', '+faststart',
    str(preview_video),
], check=True)
display(Video(str(preview_video), embed=True, width=800))
logs = sorted(Path('data/logs').glob('violations_*.csv'), key=lambda p: p.stat().st_mtime)
if logs:
    print('Latest log:', logs[-1])
    display(pd.read_csv(logs[-1]))

bundle = shutil.make_archive('/content/traffic_results', 'zip', root_dir='data', base_dir='logs')
print('Logs/evidence bundle:', bundle)
files.download(str(preview_video))
files.download(bundle)